# 01 - ResNet50

**Owner:** Caolan  |  **Plane:** sagittal  |  **Sweep task:** acl

Plain ResNet50 backbone for the architecture sweep. `He et al., CVPR 2016`.
ResNet50 + CBAM is only run later **if ResNet50 wins** the plain-backbone sweep
(single-factor ablation, see `Project Pipeline.md` -> sections 2 & 3.2).

## How to use
1. Run the **Colab bootstrap** cell.
2. Implement `src/model_factory.py` (and CBAM in `src/attention_modules.py`).
3. Train on the sagittal ACL task and report AUC.

> **GPU equivalent:** `for-gpu/run_cnn.py --backbone resnet50` (Stage 1 — backbone screening). Weights -> `model_checkpoints/`; metrics -> `for-gpu/results/`; figures -> `for-gpu/results/figures/01_backbone_screening/`.

> **Data flow:** `build_dataloaders()` -> `MRNetDataset` -> `DataLoader` -> `run_training()`. See [`../GRADING.md`](../GRADING.md) for the rubric quick-map.

In [2]:
# ============================================================
# Colab bootstrap - run this cell first.
# ============================================================
import sys
from pathlib import Path

# Mount Google Drive when running on Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

# Locate the "codes" folder (the one containing src/ and notebooks/).
if IN_COLAB:
    # TODO: set this to where you uploaded the "codes" folder on Drive.
    # CODES_DIR = Path('/content/drive/MyDrive/AI-for-MIA/codes')
    CODES_DIR = Path('/content/drive/Shareddrives/MRNet_Group3/codes')
else:
    # Local run: this notebook lives in codes/notebooks/, so go up one level.
    CODES_DIR = Path.cwd().parent

assert (CODES_DIR / 'src').exists(), (
    f"Could not find src/ under {CODES_DIR}. "
    "Edit CODES_DIR above to point to your 'codes' folder."
)

# Make `import src...` work.
if str(CODES_DIR) not in sys.path:
    sys.path.insert(0, str(CODES_DIR))

# Auto-reload edited src/*.py without restarting the kernel.
# %load_ext autoreload
# %autoreload 2

# Shared config resolves the data folder (sibling of "codes").
from src import config
print('CODES_DIR:', config.CODES_DIR)
print('DATA_DIR :', config.DATA_DIR)
assert config.DATA_DIR.exists(), (
    f"Data folder not found at {config.DATA_DIR}. Put the MRNet 'data' folder "
    "next to 'codes', or set os.environ['MRNET_DATA_DIR'] before importing config."
)
print('Setup OK - data folder found.')


Mounted at /content/drive
CODES_DIR: /content/drive/Shareddrives/MRNet_Group3/codes
DATA_DIR : /content/drive/Shareddrives/MRNet_Group3/data
Setup OK - data folder found.


In [ ]:
import torch

from src import config
from src.data_pipeline import build_dataloaders, set_seed
from src.model_factory import build_model
from src.training_utils import run_training

set_seed(config.SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_loader, val_loader = build_dataloaders(task="acl", plane=config.DEFAULT_PLANE)

model = build_model(backbone="resnet50", use_cbam=False).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5)

history = run_training(
    model, train_loader, val_loader, optimizer, scheduler, device,
    num_epochs=50,
    accumulation_steps=8,
    early_stopping_patience=10,
    checkpoint_dir=str(config.CHECKPOINTS_DIR / "resnet50_plain"),
    task_name="resnet50_acl",
)

best_val_auc = max(history["val_auc"])
print(f"Best validation AUC (ResNet50, plain, ACL, sagittal): {best_val_auc:.4f}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 106MB/s]



[resnet50_acl] Computing class weights from training data...
  Class weight | neg: 922, pos: 208, pos_weight: 4.4327


In [ ]:
# ResNet50 scaffold. Owner: Caolan.
#
# from src.data_pipeline import build_dataloaders
# from src.model_factory import build_model
# from src.training_utils import set_seed, fit
# from src.evaluation import evaluate_model
#
# TODO (sweep entry - plain backbone):
#   set_seed(config.SEED)
#   train_loader, val_loader, test_loader = build_dataloaders(config.DEFAULT_PLANE, 'acl')
#   model = build_model(backbone='resnet50', use_cbam=False)
#   fit(model, train_loader, val_loader, ...)
#   evaluate_model(model, test_loader, device)
#
# TODO (only if ResNet50 wins the sweep): rerun with use_cbam=True as the ablation.
